In [19]:
import os
import pandas as pd
import toml
from openpyxl.styles import Font, Border, Side, Alignment

# Specify config 
config_path = "configs/config_2024.toml"
config = toml.load(config_path)

# Excel formatting
no_border = Border(
    left=Side(border_style=None),
    right=Side(border_style=None),
    top=Side(border_style=None),
    bottom=Side(border_style=None)
)

In [20]:
def calc_city_vmt_shares(county, df_county_vmt):
    """Calculate VMT distribution by city for a given county in a base year

    Args:
        county (str): Name of the county to process (e.g., "King").
        df_county_vmt (df): Dataframe of county total VMT by vehicle class (index) and year (cols)
        config["base_year"]: set in config TOML; determines distributions

    Returns:
        vmt_share_df (pd.DataFrame): DataFrame with VMT shares by city and vehicle type.
    
    """

    # Load each city's results
    # Loop through each file in interzonal_vmt folder
    # Get list of files in interzonal_vmt folder
    results_df = pd.DataFrame()
    for fname in os.listdir(f"{config['output_root']}/data/city/{county}/interzonal_vmt"):
        city = fname.replace('.csv','')
        df_city = pd.read_csv(f"{config['output_root']}/data/city/{county}/interzonal_vmt/{fname}")
        df_city['Passenger Vehicles'] = df_city[['sov_vmt','hov2_vmt','hov3_vmt','tnc_vmt']].sum(axis=1)
        df_city['Medium Trucks'] = df_city['medium_truck_vmt']
        df_city['Heavy Trucks'] = df_city['heavy_truck_vmt']
        df_city['Bus'] = df_city['bus_vmt']

        total_vmt_df = pd.DataFrame(df_city.sum()[["Passenger Vehicles","Medium Trucks","Heavy Trucks","Bus"]])
        total_vmt_df['city'] = city

        results_df = pd.concat([results_df,total_vmt_df])
    results_df.rename(columns={0:config["base_year"]}, inplace=True)

    # Calculate unincorporated value as county total minus sum of cities
    df_county = pd.DataFrame(df_county_vmt[config["base_year"]])

    df_incorp_tot = results_df.groupby(results_df.index).sum()[[config["base_year"]]] 
    df = df_incorp_tot.merge(df_county, left_index=True, right_index=True, suffixes=("_incorporated_total","_county_total"))

    # Assume unicorporated area is county total minus sum of cities
    df[f"{config['base_year']}_unincorporated"] = df[f"{config['base_year']}_county_total"] - df[f"{config['base_year']}_incorporated_total"]
    df['city'] = "Unincorporated County"
    df.rename(columns={f"{config['base_year']}_unincorporated": config['base_year']}, inplace=True)

    results_df = pd.concat([results_df,df[[config["base_year"],"city"]]])

    results_df.rename(columns={f"{config['base_year']}": f"vmt_{config['base_year']}"}, inplace=True)

    results_df.reset_index(inplace=True)
    results_df.rename(columns={"index":"vehicle_type"}, inplace=True)

    # Unstack so vmt_2023 has a column for each vehicle_type
    vmt_col = f"vmt_{config['base_year']}"
    vmt_df = results_df.pivot(index='city', columns='vehicle_type', values=vmt_col).reset_index()
    vmt_df.columns.name = None

    # Calculate shares
    vmt_share_df = vmt_df.copy()
    for col in config["veh_type_map"].values():
        vmt_share_df[col] = vmt_share_df[col]/vmt_share_df[col].sum()

    # Write as Excel spreadsheet
    return vmt_share_df

In [ ]:
for county in ["King", "Kitsap", "Pierce", "Snohomish"]:
    df_county_vmt = pd.read_excel(f"{config['final_summary_dir']}/county_summary_{county}.xlsx", sheet_name='VMT', index_col=0)
    vmt_share_df = calc_city_vmt_shares(county, df_county_vmt)

    col_list = ["city"]+list(config["veh_type_map"].values())
    sheet_list = ['County VMT Share']
    # Write results to spreadsheet with two tabs: totals and shares
    with pd.ExcelWriter(f"{config['final_summary_dir']}/city_vmt_summary_{county}.xlsx") as writer:

        # calculate VMT totals within cities by applying shares to county totals, each year available
        tot_vmt_df = pd.DataFrame()
        vmt_share_df.index = vmt_share_df.city
        for year in df_county_vmt.columns:
            df_result = pd.DataFrame()
            for veh_type in ['Passenger Vehicles','Medium Trucks', 'Heavy Trucks', 'Bus']:    
                county_total = df_county_vmt[df_county_vmt.index==veh_type][year]
                df_result[veh_type] = vmt_share_df[veh_type]*county_total.values
            sheetname = f"VMT Total {year}"
            df_result.to_excel(writer, sheet_name=sheetname, index=True)
            sheet_list.append(sheetname)

        # Write VMT Shares once since it does not change
        vmt_share_df[col_list].to_excel(writer, sheet_name='County VMT Share', index=False)

        # Apply formatting
        for sheetname in sheet_list:
            ws = writer.sheets[sheetname]
            for row in ws.iter_rows():
                for cell in row:
                    cell.border = no_border
                    cell.font = Font(bold=False)
                    if cell.column == 1:
                        cell.alignment = Alignment(horizontal='center')
                    if isinstance(cell.value, (int, float)):
                        if sheetname == "County VMT Share":
                            cell.number_format = '0.0000'
                        else:
                            cell.number_format = '#,##0'

            # Set column widths based on header text length
            for col in ws.columns:
                header = col[0].value
                width = max(len(str(header)) if header else 25, 25) + 2

                ws.column_dimensions[col[0].column_letter].width = width

# Compile Emissions for Cities
# Use VMT shares by vehicle type to calculate city emissions from county totals
df_county_emissions = pd.read_excel(f"{config['final_summary_dir']}/county_summary_{county}.xlsx", sheet_name='Total Emissions', index_col=0)

with pd.ExcelWriter(f"{config['final_summary_dir']}/city_emissions_summary_{county}.xlsx") as writer:
    emissions_df = pd.DataFrame()
    vmt_share_df.index = vmt_share_df.city
    for year in config["analysis_year_list"]:
        df_result = pd.DataFrame()
        for veh_type in ['Passenger Vehicles','Medium Trucks', 'Heavy Trucks', 'Bus']:    
            county_total = df_county_emissions[df_county_emissions.index==veh_type][year]
            df_result[veh_type] = vmt_share_df[veh_type]*county_total.values
        df_result.to_excel(writer, sheet_name=f'CO2e {year}', index=True)

        # Apply formatting
        ws = writer.sheets[f'CO2e {year}']
        for row in ws.iter_rows():
            for cell in row:
                cell.border = no_border
                cell.font = Font(bold=False)
                if cell.column == 1:
                    cell.alignment = Alignment(horizontal='center')
                if isinstance(cell.value, (int, float)):
                    cell.number_format = '#,##0.0'

        # Set column widths based on header text length
        for col in ws.columns:
            header = col[0].value

            width = max(len(str(header)) if header else 25, 25) + 2            
            ws.column_dimensions[col[0].column_letter].width = width

KeyboardInterrupt: 